# Evaluate customer review sentiment
We reuse a pretrained model and evaluate its transfer to Amazon reviews. Run the download and evaluation scripts in the README first. This notebook reads their artifacts; it does not train a model.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'data/development.csv').exists(), 'Run python -m scripts.download_data first'
development = pd.read_csv(ROOT / 'data/development.csv')
display(development.head(10))

ModuleNotFoundError: No module named 'pandas'

## Explore development data
Use development data to inspect examples before looking at evaluation results. Labels: 0 negative, 1 positive. A balanced sample is a design choice, not a measurement of customer satisfaction.

In [ ]:
display(development['label'].value_counts().sort_index())
display(development['text'].str.len().describe())
development['text'].str.len().hist(bins=20)
plt.xlabel('Review length (characters)')
plt.ylabel('Reviews')
plt.show()

## Model versus baseline
Accuracy counts correct predictions. Precision measures correctness among predictions for a class; recall measures coverage of actual examples of that class. F1 balances the two. The baseline predicts one class for every review. Model scores are not calibrated reliability estimates.

In [ ]:
assert (ROOT / 'reports/metrics.json').exists(), 'Run python -m scripts.evaluate first'
metrics = json.loads((ROOT / 'reports/metrics.json').read_text())
display(pd.Series({k: metrics[k] for k in ['rows', 'accuracy', 'baseline_accuracy', 'truncated_reviews', 'inference_seconds']}))
display(pd.DataFrame(metrics['classification_report']).T)
display(Image(filename=str(ROOT / 'reports/confusion_matrix.png')))

## Inspect errors
Read the expected and predicted sentiment alongside the review. Identify mixed sentiment, sarcasm, rating disagreement, or truncation. Do high model scores always mean correct predictions?

In [ ]:
predictions = pd.read_csv(ROOT / 'reports/predictions.csv')
errors = predictions[predictions['expected_sentiment'] != predictions['predicted_sentiment']]
display(errors.sort_values('score', ascending=False).head(10))

## Conclusions to write yourself
1. How much better is the model than the baseline?
2. Which class is harder and what examples support that?
3. What does the confusion matrix tell a webshop owner?
4. Why does this balanced, buffered sample not establish production performance?
5. How could repeated tuning on this evaluation sample bias our reported result?
6. What would you collect next before using it for business decisions?

Source: [Amazon Polarity](https://huggingface.co/datasets/fancyzhx/amazon_polarity). Model: [DistilBERT SST-2](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english). Model and data revisions are stored in the evaluation report.